## orflib Library Examples

In [1]:
import orflib as orf
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
plt.style.use('ggplot')

print("orflib version: {0}".format(orf.version()))
print("pid: {0}".format(os.getpid()))

orflib version: 0.3.0-debug
pid: 16672


In [2]:
name = "World"
print(orf.sayHello(name))

Hello World!


In [3]:
x = [1, 2, 3]
y = [4, 5]
op = orf.outerProd(x, y)
print(f'x: {x}\ny: {y}')
print(f'outerProd:\n{op}')

x: [1, 2, 3]
y: [4, 5]
outerProd:
[[ 4.  5.]
 [ 8. 10.]
 [12. 15.]]


In [4]:
# Matrix Echo
m = np.array([[1, 2, 3], [4, 5, 6]])
em = orf.echoMatrix(m)
print(f'orig matrix:\n{m}')
print(f'echo matrix:\n{em}')

orig matrix:
[[1 2 3]
 [4 5 6]]
echo matrix:
[[1. 2. 3.]
 [4. 5. 6.]]


### Analytic Prices

In [6]:
#fwdPrice
fwdpx = orf.fwdPrice(spot = 100, timetoexp = 1.0, intrate = 0.02, divyield = 0.04)
print('Forward price analytic solution')
print(f'Price={fwdpx:.4f}')

Forward price analytic solution
Price=98.0199


In [7]:
#digiBS
digi = orf.digiBS(payofftype = 1, spot = 100, timetoexp = 1.0, strike = 100,
                  intrate = 0.02, divyield = 0.04, volatility = 0.2)
print('Digital call using Black-Scholes analytic solution')
print(f"European Call Price:{round(digi[0],4)}\nDelta:{round(digi[1],4)}\nGamma:{round(digi[2],20)}\nTheta:{round(digi[3],4)}\nVega:{round(digi[4],20)}")

Digital call using Black-Scholes analytic solution
European Call Price:0.4124
Delta:0.0192
Gamma:2.1e-19
Theta:0.0466
Vega:4.2555e-16


In [8]:
#euroBS
euro = orf.euroBS(payofftype = 1, spot = 100, timetoexp = 1.0, strike = 100,
                  intrate = 0.02, divyield = 0.04, volatility = 0.4)
print('European call using Black-Scholes analytic solution')
print(f"European Call Price:{round(euro[0],4)}\nDelta:{round(euro[1],4)}\nGamma:{round(euro[2],4)}\nTheta:{round(euro[3],4)}\nVega:{round(euro[4],4)}")

European call using Black-Scholes analytic solution
European Call Price:14.4327
Delta:0.5377
Gamma:0.0095
Theta:-6.2162
Vega:37.9012


### Homework Problem 3

In [9]:
np.random.seed(42)  

def random_params(n=10):
    payoff = np.random.choice([+1, -1], size=n)           # +1 call, -1 put
    S      = np.random.uniform(100, 110, n)               # spot
    K      = np.random.uniform(100, 110, n)               # strike
    r      = np.random.uniform(0.01, 0.03, n)             # interest rate
    q      = np.random.uniform(0.0, 0.03, n)              # dividend yield
    timeToExp    = np.random.uniform(0.0, 1.5, n)         # time to expiry (years)
    sigma  = np.random.uniform(0.2, 0.3, n)               # volatility
    return payoff, S, K, timeToExp, r, q, sigma

def pde_diff(Price, Delta, Gamma, Theta, S, r, q, sigma):
    return Theta + (r - q) * S * Delta + 0.5 * sigma**2 * S**2 * Gamma - r * Price

# -------- Part 1: European options --------
payoff, S, K, timeToExp, r, q, sigma = random_params(10)

rows = []
for i in range(10):
    P, d, g, th, vega = orf.euroBS(payoff[i], S[i], K[i], timeToExp[i], r[i], q[i], sigma[i])
    diff = pde_diff(P, d, g, th, S[i], r[i], q[i], sigma[i])
    rows.append({
        "Payoff": payoff[i],
        "Spot": float(S[i]),
        "Strike": float(K[i]),
        "TimeToExp": float(timeToExp[i]),
        "IntRate": float(r[i]),
        "DivYield": float(q[i]),
        "Volatility": float(sigma[i]),
        "Price": float(P),
        "Delta": float(d),
        "Gamma": float(g),
        "Theta": float(th),
        "Diff": float(diff),
    })

euro_df = pd.DataFrame(rows)
print("European options — max |Diff|:", np.abs(euro_df["Diff"]).max())
display(euro_df)

# -------- Part 2: Digital (cash-or-nothing) options --------
rows_d = []
for i in range(10):
    P, d, g, th, vega = orf.digiBS(payoff[i], S[i], K[i], timeToExp[i], r[i], q[i], sigma[i])
    diff = pde_diff(P, d, g, th, S[i], r[i], q[i], sigma[i])
    rows_d.append({
        "Payoff": payoff[i],
        "Spot": float(S[i]),
        "Strike": float(K[i]),
        "TimeToExp": float(timeToExp[i]),
        "IntRate": float(r[i]),
        "DivYield": float(q[i]),
        "Volatility": float(sigma[i]),
        "Price": float(P),
        "Delta": float(d),
        "Gamma": float(g),
        "Theta": float(th),
        "Diff": float(diff),
    })

digi_df = pd.DataFrame(rows_d)
print("Digital options — max |Diff|:", np.abs(digi_df["Diff"]).max())
display(digi_df)

European options — max |Diff|: 1.582067810090848e-15


,Payoff,Spot,Strike,TimeToExp,IntRate,DivYield,Volatility,Price,Delta,Gamma,Theta,Diff
0,1,101.559945,101.834045,0.993783,0.025704,0.024252,0.292187,11.425110,0.542954,0.013034,-5.525077,4.440892e-16
1,-1,100.580836,103.042422,0.467567,0.013993,0.009138,0.208849,6.941000,-0.530410,0.027563,-5.725073,-4.440892e-16
2,1,108.661761,105.247564,0.780102,0.020285,0.002930,0.219598,10.821942,0.628382,0.017877,-6.054934,1.582068e-15
3,1,106.011150,104.319450,0.820065,0.021848,0.020527,0.204523,8.545700,0.563934,0.019639,-4.508449,-5.273559e-16
4,1,107.080726,102.912291,0.277282,0.010929,0.013205,0.232533,7.410430,0.645786,0.028200,-8.503692,-1.457168e-15
5,-1,100.205845,106.118529,1.454377,0.022151,0.003661,0.238868,13.196211,-0.482123,0.013737,-2.749562,-4.996004e-16
6,1,109.699099,101.394939,1.162699,0.013410,0.014855,0.227135,14.609948,0.657492,0.013265,-3.817637,-1.443290e-15
7,1,108.324426,102.921446,1.409248,0.011301,0.001032,0.282874,17.664017,0.640901,0.010252,-5.326193,8.881784e-16
8,1,102.123391,103.663618,1.342241,0.028978,0.027280,0.235675,10.142073,0.516607,0.013737,-3.774397,7.216450e-16
9,-1,101.818250,104.560700,0.896850,0.029313,0.007763,0.228093,9.100937,-0.467070,0.017963,-3.552735,2.220446e-16


Digital options — max |Diff|: 1.6653345369377348e-16


,Payoff,Spot,Strike,TimeToExp,IntRate,DivYield,Volatility,Price,Delta,Gamma,Theta,Diff
0,1,101.559945,101.834045,0.993783,0.025704,0.024252,0.292187,0.429299,0.012999,-0.000062,0.036463,-6.591949e-17
1,-1,100.580836,103.042422,0.467567,0.013993,0.009138,0.208849,0.585099,-0.026904,-0.000154,0.055217,-1.561251e-16
2,1,108.661761,105.247564,0.780102,0.020285,0.002930,0.219598,0.545942,0.018457,-0.000290,0.058887,1.665335e-16
3,1,106.011150,104.319450,0.820065,0.021848,0.020527,0.204523,0.491160,0.019958,-0.000188,0.052210,-6.938894e-18
4,1,107.080726,102.912291,0.277282,0.010929,0.013205,0.232533,0.599936,0.029342,-0.000851,0.277567,-1.422473e-16
5,-1,100.205845,106.118529,1.454377,0.022151,0.003661,0.238868,0.579614,-0.012972,0.000017,0.031933,-3.816392e-17
6,1,109.699099,101.394939,1.162699,0.013410,0.014855,0.227135,0.567250,0.014352,-0.000233,0.082345,9.887924e-17
7,1,108.324426,102.921446,1.409248,0.011301,0.001032,0.282874,0.502919,0.010790,-0.000108,0.044281,7.719519e-17
8,1,102.123391,103.663618,1.342241,0.028978,0.027280,0.235675,0.411095,0.013533,-0.000044,0.022223,2.949030e-17
9,-1,101.818250,104.560700,0.896850,0.029313,0.007763,0.228093,0.541859,-0.017492,0.000059,0.038298,-7.285839e-17


We see that the errors are extremely small and of the order 10^-15 for this problem.